In [1]:
import os

In [2]:
# Set this to  actual project path
project_path = r"c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline"

os.chdir(project_path)

print(f"Now at: {os.getcwd()}")

Now at: c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline


In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [4]:
from src.TextInsightMlopsPipeline.constants import *
from src.TextInsightMlopsPipeline.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import pandas as pd
from tqdm import tqdm

c:\Users\saqib\OneDrive\Desktop\text-insight-mlops-pipeline\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
!pip install evaluate 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import nltk
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

nltk.download("punkt", quiet=True)

True

In [8]:
import evaluate



class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer, 
                               batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu", 
                               column_text="article", 
                               column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):
            
            inputs = tokenizer(article_batch, max_length=1024,  truncation=True, 
                            padding="max_length", return_tensors="pt")
            
            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                            attention_mask=inputs["attention_mask"].to(device), 
                            length_penalty=0.8, num_beams=8, max_length=128)
            ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''
            
            # Finally, we decode the generated texts, 
            # replace the  token, and add the decoded texts with the references to the metric.
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                                    clean_up_tokenization_spaces=True) 
                for s in summaries]      
            
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
            
            
            metric.add_batch(predictions=decoded_summaries, references=target_batch)
            
        #  Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
       
        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)


        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        rouge_metric = evaluate.load('rouge')

        #rouge_metric = rouge_metric

        score = self.calculate_metric_on_test_ds(
        dataset_samsum_pt['test'][0:10], rouge_metric, model_pegasus, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
            )

        # Directly use the scores without accessing fmeasure or mid
        rouge_dict = {rn: score[rn] for rn in rouge_names}

        df = pd.DataFrame(rouge_dict, index = ['pegasus'] )
        df.to_csv(self.config.metric_file_name, index=False)

In [9]:
!pip install rouge_score absl-py nltk -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# The local evaluation is skipped because loading the 2.28GB PEGASUS model
# freezes the machine. Evaluation was run on Google Colab instead.
# ROUGE scores are saved in artifacts/model_evaluation/metrics.csv

# config = ConfigurationManager()
# model_evaluation_config = config.get_model_evaluation_config()
# model_evaluation = ModelEvaluation(config=model_evaluation_config)
# model_evaluation.evaluate()

# ─── Colab Evaluation Steps ───────────────────────────────────────────────────

# Cell 1 — Install dependencies
# !pip install transformers==4.51.0 datasets evaluate rouge_score absl-py nltk -q

# Cell 2 — Download dataset
# !wget -q https://github.com/krishnaik06/datasets/raw/refs/heads/main/summarizer-data.zip
# !unzip -q -o summarizer-data.zip

# Cell 3 — Mount Google Drive (make sure tokenizer and model are saved there)
# from google.colab import drive
# drive.mount('/content/drive')

# Cell 4 — Load model on CPU with float32 (float16 produces empty predictions)
# import torch, gc
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# from datasets import load_from_disk
# torch.cuda.empty_cache()
# gc.collect()
# device = "cpu"
# tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/tokenizer")
# model = AutoModelForSeq2SeqLM.from_pretrained(
#     "/content/drive/MyDrive/pegasus-samsum-model",
#     torch_dtype=torch.float32
# ).to(device)
# dataset = load_from_disk("samsum_dataset")

# Cell 5 — Compute ROUGE scores
# import evaluate, pandas as pd
# rouge_metric = evaluate.load("rouge")
# rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
# for dialogue, summary in zip(dataset["test"][0:3]["dialogue"], dataset["test"][0:3]["summary"]):
#     inputs = tokenizer(dialogue, max_length=1024, truncation=True, return_tensors="pt")
#     with torch.no_grad():
#         summary_ids = model.generate(
#             inputs["input_ids"],
#             attention_mask=inputs["attention_mask"],
#             length_penalty=0.8, num_beams=2, max_length=128
#         )
#     decoded = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
#     rouge_metric.add_batch(predictions=[decoded], references=[summary])
# score = rouge_metric.compute()
# rouge_dict = {rn: score[rn] for rn in rouge_names}
# df = pd.DataFrame(rouge_dict, index=["pegasus"])
# print(df)
# df.to_csv("metrics.csv", index=False)

# Cell 6 — Download results
# from google.colab import files
# files.download("metrics.csv")

# Results achieved:
# rouge1: 0.4300, rouge2: 0.1858, rougeL: 0.3558, rougeLsum: 0.3558